# Notebook 3 – Categorical Feature Engineering

## Objective

Categorical Feature Engineering is the process of converting categorical information into useful representations that Machine Learning models can understand.

Topics covered:

- Categorical Variables
- Nominal Variables
- Ordinal Variables
- Binary Variables
- One-Hot Encoding
- Ordinal Encoding
- Label Encoding
- Frequency Encoding
- Target Encoding
- Rare Category Grouping
- Category Cardinality
- Handling Unknown Categories
- Combining Categories

The objective is to understand the difference between these techniques and when each technique should be used.

Dataset: Titanic-Dataset.csv
Target Variable: Survived

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Titanic-Dataset.csv")

print("Dataset Shape:", df.shape)
print(df[["Sex", "Embarked", "Pclass", "Name"]].head())

Dataset Shape: (891, 12)
      Sex Embarked  Pclass                                               Name
0    male        S       3                            Braund, Mr. Owen Harris
1  female        C       1  Cumings, Mrs. John Bradley (Florence Briggs Th...
2  female        S       3                             Heikkinen, Miss. Laina
3  female        S       1       Futrelle, Mrs. Jacques Heath (Lily May Peel)
4    male        S       3                           Allen, Mr. William Henry


# 1. Categorical Variables

Categorical variables contain values that represent groups or categories rather than continuous numerical measurements.

Examples from the Titanic dataset:

- Sex
- Embarked
- Pclass
- Cabin
- Ticket

Categorical variables need suitable representation before being given to many Machine Learning algorithms.

There are three important types:

- Nominal
- Ordinal
- Binary

In [2]:
categorical_columns = ["Sex", "Embarked", "Pclass"]

for column in categorical_columns:
    print(f"\n{column}:")
    print(df[column].value_counts(dropna=False))


Sex:
Sex
male      577
female    314
Name: count, dtype: int64

Embarked:
Embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64

Pclass:
Pclass
3    491
1    216
2    184
Name: count, dtype: int64


# 2. Nominal Variables

Nominal variables contain categories that do not have a natural order.

### Titanic Examples

- Sex: Male, Female
- Embarked: C, Q, S

There is no meaningful order such as C > Q > S.

### Suitable Encoding

One-Hot Encoding is generally suitable for nominal variables.

### When to Use

Use One-Hot Encoding when categories are independent and have no natural ranking.

# 3. Ordinal Variables

Ordinal variables contain categories with a meaningful order or ranking.

Example:

Low < Medium < High

For the Titanic dataset, Pclass can be treated as an ordinal feature because passenger classes have an ordered structure:

1st Class > 2nd Class > 3rd Class

### Suitable Encoding

Ordinal Encoding can be used when the ordering is meaningful.

### Important Note

Ordinal encoding should not be used for nominal categories because assigning numbers can incorrectly suggest an order.

In [3]:
print("Pclass categories:")
print(sorted(df["Pclass"].dropna().unique()))

Pclass categories:
[np.int64(1), np.int64(2), np.int64(3)]


# 4. Binary Variables

A binary variable has only two possible categories.

### Titanic Example

Sex contains two main categories:

- Male
- Female

A binary feature can therefore be represented using 0 and 1.

### When to Use

Binary encoding is useful when a categorical variable contains exactly two meaningful categories.

### Required Documentation

**Feature Name:** Sex_Binary

**Source Columns:** Sex

**Logic:** Female = 1 and Male = 0.

**Reason:** Convert a binary categorical variable into numerical form.

**Business Meaning:** Represents the passenger's sex as a binary value.

**ML Relevance:** Allows numerical Machine Learning algorithms to use the information.

**Leakage Check:** No. The feature does not use Survived.

**Final Decision:** Retain

In [4]:
df["Sex_Binary"] = df["Sex"].map({
    "male": 0,
    "female": 1
})

print(df[["Sex", "Sex_Binary"]].head())

      Sex  Sex_Binary
0    male           0
1  female           1
2  female           1
3  female           1
4    male           0


# 5. One-Hot Encoding

One-Hot Encoding creates a separate binary feature for each category.

### Example

Embarked contains:

- C
- Q
- S

It can become:

- Embarked_C
- Embarked_Q
- Embarked_S

### When to Use

Use One-Hot Encoding mainly for nominal categorical variables with a manageable number of categories.

### Advantages

- Does not create artificial ordering
- Easy to interpret
- Suitable for nominal variables

### Limitation

High-cardinality variables can create many new columns.

In [5]:
one_hot = pd.get_dummies(
    df["Embarked"],
    prefix="Embarked",
    dtype=int
)

print(one_hot.head())

   Embarked_C  Embarked_Q  Embarked_S
0           0           0           1
1           1           0           0
2           0           0           1
3           0           0           1
4           0           0           1


# 6. Required Documentation – One-Hot Encoding

**Feature Name:** Embarked_C, Embarked_Q, Embarked_S

**Source Columns:** Embarked

**Logic:** Create one binary column for each category.

**Reason:** Convert a nominal categorical variable into numerical features without introducing artificial ordering.

**Business Meaning:** Each feature indicates whether the passenger embarked from a particular location.

**ML Relevance:** Allows Machine Learning models to use Embarked information numerically.

**Leakage Check:** No. The encoding uses only the source categorical variable.

**Final Decision:** Retain

# 7. Ordinal Encoding

Ordinal Encoding assigns numerical values according to the natural order of categories.

### Example

Low → 0
Medium → 1
High → 2

### Titanic Example

Pclass can be represented according to its class order.

### When to Use

Use Ordinal Encoding when:

- Categories have a meaningful order
- The numerical ordering represents real domain meaning

### Risk

Do not use ordinal encoding for nominal variables because the numbers may create a false relationship between categories.

In [6]:
pclass_ordinal = df["Pclass"].map({
    3: 0,
    2: 1,
    1: 2
})

print(pd.DataFrame({
    "Pclass": df["Pclass"],
    "Pclass_Ordinal": pclass_ordinal
}).head())

   Pclass  Pclass_Ordinal
0       3               0
1       1               2
2       3               0
3       1               2
4       3               0


# 8. Required Documentation – Ordinal Encoding

**Feature Name:** Pclass_Ordinal

**Source Columns:** Pclass

**Logic:** 3rd class = 0, 2nd class = 1, 1st class = 2.

**Reason:** Represent the ordered structure of passenger class numerically.

**Business Meaning:** Represents the relative passenger class level.

**ML Relevance:** Allows models to use the ordered information in Pclass.

**Leakage Check:** No. Survived is not used.

**Final Decision:** Retain

# 9. Label Encoding

Label Encoding assigns an integer to each category.

Example:

Category A → 0
Category B → 1
Category C → 2

### Important Difference

Label Encoding can introduce an artificial numerical order when used on nominal input features.

Therefore, it is generally more appropriate for encoding a target variable or for categories where the assigned ordering is acceptable.

### Titanic Example

The target `Survived` already contains binary values:

0 = Did not survive
1 = Survived

For a binary target, additional label encoding is not required.

In [7]:
print("Target values:")
print(df["Survived"].unique())

Target values:
[0 1]


# 10. Frequency Encoding

Frequency Encoding replaces each category with the number or proportion of records belonging to that category.

### Formula

Frequency = Number of records in category / Total number of records

### Titanic Example

Frequency encoding can be applied to Ticket.

### When to Use

Useful when:

- A categorical variable has many unique categories
- One-hot encoding would create too many columns
- Category frequency may contain useful information

### Risk

Frequency represents how common a category is, but it does not necessarily represent its meaning.

In [8]:
ticket_frequency = df["Ticket"].value_counts(normalize=True)

df["Ticket_Frequency"] = df["Ticket"].map(ticket_frequency)

print(df[["Ticket", "Ticket_Frequency"]].head())

             Ticket  Ticket_Frequency
0         A/5 21171          0.001122
1          PC 17599          0.001122
2  STON/O2. 3101282          0.001122
3            113803          0.002245
4            373450          0.001122


# 11. Required Documentation – Frequency Encoding

**Feature Name:** Ticket_Frequency

**Source Columns:** Ticket

**Logic:** Replace each Ticket category with its proportion in the dataset.

**Reason:** Ticket has relatively high cardinality, so frequency encoding provides numerical information without creating a large number of columns.

**Business Meaning:** Represents how common a particular ticket value is in the dataset.

**ML Relevance:** Category frequency may provide useful information about shared or common ticket groups.

**Leakage Check:** The frequency should be calculated from training data only when building a real ML model. Calculating it from the complete dataset before splitting can cause information leakage.

**Final Decision:** Needs Further Analysis

# 12. Target Encoding

Target Encoding replaces a category with a statistic calculated from the target variable.

For classification, a common approach is:

Category Target Mean = Average Target Value for that Category

### Example

For Sex:

Mean Survived for Male
Mean Survived for Female

### Advantage

It can represent the relationship between a category and the target using a single numerical feature.

### Important Risk

Target Encoding can easily cause data leakage.

The target information must be calculated using training data only, preferably with an out-of-fold approach during training.

It should never be calculated using the complete dataset before train-test splitting.

In [9]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["Survived"]
)

target_mean = train_df.groupby("Sex")["Survived"].mean()

train_df["Sex_Target_Encoded"] = train_df["Sex"].map(target_mean)
test_df["Sex_Target_Encoded"] = test_df["Sex"].map(target_mean)

print(train_df[["Sex", "Survived", "Sex_Target_Encoded"]].head())
print("\nTest data:")
print(test_df[["Sex", "Sex_Target_Encoded"]].head())

        Sex  Survived  Sex_Target_Encoded
692    male         1            0.185185
481    male         0            0.185185
527    male         0            0.185185
855  female         1            0.743083
801  female         1            0.743083

Test data:
        Sex  Sex_Target_Encoded
565    male            0.185185
160    male            0.185185
553    male            0.185185
860    male            0.185185
241  female            0.743083


# 13. Required Documentation – Target Encoding

**Feature Name:** Sex_Target_Encoded

**Source Columns:** Sex and Survived during training

**Logic:** Replace each Sex category with its mean Survived value calculated from the training data.

**Reason:** Capture the relationship between the categorical variable and the target using a numerical representation.

**Business Meaning:** Represents the historical survival rate associated with each Sex category in the training data.

**ML Relevance:** Can provide a strong predictive signal in some datasets.

**Leakage Check:** High leakage risk. Target statistics must be calculated using training data only and handled carefully during cross-validation.

**Final Decision:** Needs Further Analysis

# 14. Rare Category Grouping

Rare Category Grouping combines categories that occur very infrequently into a common category such as `Rare`.

### When to Use

Useful when:

- A categorical feature contains many low-frequency categories
- Rare categories may cause sparse features
- Some categories do not have enough observations for reliable learning

### Titanic Example

Passenger titles extracted from Name can contain many uncommon titles.

Rare titles can be grouped together.

This reduces unnecessary category complexity while preserving the information that the category is uncommon.

In [10]:
df["Title"] = df["Name"].str.extract(
    r",\s*([^.]*)\.",
    expand=False
).str.strip()

title_counts = df["Title"].value_counts()

rare_titles = title_counts[title_counts < 10].index

df["Title_Grouped"] = df["Title"].where(
    ~df["Title"].isin(rare_titles),
    "Rare"
)

print(df[["Title", "Title_Grouped"]].head(10))
print("\nGrouped title frequencies:")
print(df["Title_Grouped"].value_counts())

    Title Title_Grouped
0      Mr            Mr
1     Mrs           Mrs
2    Miss          Miss
3     Mrs           Mrs
4      Mr            Mr
5      Mr            Mr
6      Mr            Mr
7  Master        Master
8     Mrs           Mrs
9     Mrs           Mrs

Grouped title frequencies:
Title_Grouped
Mr        517
Miss      182
Mrs       125
Master     40
Rare       27
Name: count, dtype: int64


# 15. Required Documentation – Rare Category Grouping

**Feature Name:** Title_Grouped

**Source Columns:** Name

**Logic:** Extract the passenger title from Name and group titles with fewer than 10 occurrences into `Rare`.

**Reason:** Reduce the number of low-frequency categories.

**Business Meaning:** Represents passenger titles while grouping uncommon titles together.

**ML Relevance:** Can reduce category complexity and create more stable categorical representations.

**Leakage Check:** No. The feature is created from Name only.

**Final Decision:** Retain

# 16. Category Cardinality

Cardinality means the number of unique categories in a categorical variable.

Examples:

- Sex → Low cardinality
- Embarked → Low cardinality
- Ticket → Higher cardinality
- Name → High cardinality
- Cabin → High cardinality

### Why Cardinality Matters

High-cardinality features can create:

- Too many encoded columns
- Sparse data
- Higher memory usage
- Increased model complexity
- Overfitting risk

### Engineering Decision

The encoding technique should depend partly on category cardinality.

In [11]:
categorical_columns = ["Sex", "Embarked", "Ticket", "Cabin", "Name"]

cardinality = pd.DataFrame({
    "Feature": categorical_columns,
    "Unique_Categories": [
        df[column].nunique(dropna=True)
        for column in categorical_columns
    ]
})

print(cardinality)

    Feature  Unique_Categories
0       Sex                  2
1  Embarked                  3
2    Ticket                681
3     Cabin                147
4      Name                891


# 17. Handling Unknown Categories

An unknown category occurs when a category appears during prediction that was not present when the encoder was fitted.

This can happen when:

- New data contains a new category
- Categories change over time
- Training and production data contain different categories

### Recommended Approach

For One-Hot Encoding, `handle_unknown="ignore"` allows unseen categories to be handled without producing an error.

### AI/ML Relevance

Handling unknown categories makes a preprocessing workflow more robust for real-world prediction data.

In [12]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

train_encoded = encoder.fit_transform(train_df[["Embarked"]])

print("Encoded Shape:", train_encoded.shape)
print("Encoded Categories:", encoder.categories_)

Encoded Shape: (712, 4)
Encoded Categories: [array(['C', 'Q', 'S', nan], dtype=object)]


# 18. Required Documentation – Handling Unknown Categories

**Feature Name:** Embarked_Encoded

**Source Columns:** Embarked

**Logic:** One-Hot Encode Embarked using `handle_unknown="ignore"`.

**Reason:** Prevent errors when a new category appears in future data.

**Business Meaning:** Represents the passenger's embarkation location.

**ML Relevance:** Makes the categorical preprocessing process robust to unseen categories.

**Leakage Check:** No target information is used.

**Final Decision:** Retain

# 19. Combining Categories

Combining Categories means merging related categories into a smaller number of meaningful groups.

### Titanic Example

Passenger titles can be grouped into broader categories such as:

- Mr
- Mrs
- Miss
- Master
- Other

### When to Use

Use category combining when:

- Several categories have similar meanings
- Some categories are too rare
- Domain knowledge supports combining them
- Reducing categories can simplify the model

### Important Rule

Categories should be combined based on a meaningful business or domain reason, not randomly.

In [13]:
def group_title(title):
    if title == "Mr":
        return "Mr"
    elif title == "Mrs":
        return "Mrs"
    elif title == "Miss":
        return "Miss"
    elif title == "Master":
        return "Master"
    else:
        return "Other"

df["Title_Combined"] = df["Title"].apply(group_title)

print(df[["Title", "Title_Combined"]].head(10))
print("\nCombined categories:")
print(df["Title_Combined"].value_counts())

    Title Title_Combined
0      Mr             Mr
1     Mrs            Mrs
2    Miss           Miss
3     Mrs            Mrs
4      Mr             Mr
5      Mr             Mr
6      Mr             Mr
7  Master         Master
8     Mrs            Mrs
9     Mrs            Mrs

Combined categories:
Title_Combined
Mr        517
Miss      182
Mrs       125
Master     40
Other      27
Name: count, dtype: int64


# 20. Required Documentation – Combining Categories

**Feature Name:** Title_Combined

**Source Columns:** Name

**Logic:** Extract passenger titles and combine less common titles into `Other`.

**Reason:** Reduce unnecessary categorical complexity while preserving major title groups.

**Business Meaning:** Represents broad passenger title groups.

**ML Relevance:** Reduces the number of categories and may provide a more stable representation for Machine Learning.

**Leakage Check:** No. The feature uses passenger Name information and does not use Survived.

**Final Decision:** Retain

# 21. Difference Between Categorical Encoding Techniques

| Technique | Best Used For | Main Idea | Important Risk |
|---|---|---|---|
| One-Hot Encoding | Nominal, low/medium cardinality | Creates binary columns | High dimensionality |
| Ordinal Encoding | Ordered categories | Assigns ordered numerical values | Incorrect order if used wrongly |
| Label Encoding | Usually target variables | Assigns integer labels | Artificial order for nominal inputs |
| Frequency Encoding | High-cardinality categories | Uses category frequency | Frequency may not represent meaning |
| Target Encoding | High-cardinality categories | Uses target statistic | High leakage risk |
| Rare Category Grouping | Rare categories | Groups uncommon categories | May lose useful information |
| Category Combining | Related categories | Combines meaningful groups | Poor grouping can lose information |

### Main Principle

There is no single encoding technique that is best for every categorical variable.

The correct technique depends on:

- Variable type
- Cardinality
- Domain meaning
- Model type
- Data size
- Leakage risk

# 22. Final Feature Engineering Documentation Summary

| Feature Name | Source Columns | Logic | Reason | Business Meaning | ML Relevance | Leakage Check | Final Decision |
|---|---|---|---|---|---|---|---|
| Sex_Binary | Sex | Female = 1, Male = 0 | Convert binary category to numerical form | Passenger sex | Usable by ML models | No | Retain |
| Embarked_C/Q/S | Embarked | One-hot encoding | Represent nominal categories | Embarkation location | Useful categorical representation | No | Retain |
| Pclass_Ordinal | Pclass | 3 → 0, 2 → 1, 1 → 2 | Preserve meaningful order | Passenger class level | Represents ordered information | No | Retain |
| Ticket_Frequency | Ticket | Category frequency | Handle higher cardinality | Ticket commonness | Provides category frequency information | Calculate from training data only | Needs Further Analysis |
| Sex_Target_Encoded | Sex + Survived | Training target mean | Capture category-target relationship | Historical survival rate by Sex | Potentially strong predictive feature | High leakage risk | Needs Further Analysis |
| Title_Grouped | Name | Rare titles → Rare | Reduce rare categories | Passenger title group | More stable categorical representation | No | Retain |
| Embarked_Encoded | Embarked | One-hot with unknown handling | Handle unseen categories | Embarkation location | Robust ML representation | No | Retain |
| Title_Combined | Name | Major titles retained, others → Other | Reduce category complexity | Broad passenger title group | Simplifies categorical information | No | Retain |

# 23. Conclusion

Categorical Feature Engineering converts categorical information into representations that Machine Learning models can use.

In this notebook, we covered:

- Categorical Variables
- Nominal Variables
- Ordinal Variables
- Binary Variables
- One-Hot Encoding
- Ordinal Encoding
- Label Encoding
- Frequency Encoding
- Target Encoding
- Rare Category Grouping
- Category Cardinality
- Handling Unknown Categories
- Combining Categories

The most important engineering decision is to select the encoding technique according to the type and characteristics of the categorical variable.

For example:

- Nominal → One-Hot Encoding
- Ordered categories → Ordinal Encoding
- Binary → Binary Encoding
- High cardinality → Frequency or carefully applied Target Encoding
- Rare categories → Rare Category Grouping
- Related categories → Combining Categories

Target Encoding requires special attention because it uses target information and therefore has a high risk of data leakage.

The final goal is not to encode every categorical column automatically, but to create representations that are meaningful, reliable, and useful for Machine Learning.